In [1]:
import pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('train.csv')

df =df.drop(columns=['id', 'diet_type', 'gender'])
df.head(3)

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,stress_level,sleep_quality,physical_activity_level,smoking_alcohol
0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,high,average,sedentary,yes
1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,low,average,moderate,yes
2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,high,poor,active,yes


In [4]:
print("⏳ Step 1: Poore dataset (df) ke Missing values (NaN) ko clean kiya ja raha hai...")
for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        df[col] = df[col].fillna(df[col].mean())
    elif df[col].dtype in ['object', 'string']:
        if not df[col].mode().empty:
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna('Missing')
    else:
        print(f"⚠️ Column '{col}' ka dtype alag hai: {df[col].dtype}")

print("✅ df ka poora data bilkul saaf ho gaya! Target column bhi clean hai.\n")
print(df.isnull().sum())

⏳ Step 1: Poore dataset (df) ke Missing values (NaN) ko clean kiya ja raha hai...
✅ df ka poora data bilkul saaf ho gaya! Target column bhi clean hai.

health_condition           0
sleep_duration             0
heart_rate                 0
bmi                        0
calorie_expenditure        0
step_count                 0
exercise_duration          0
water_intake               0
stress_level               0
sleep_quality              0
physical_activity_level    0
smoking_alcohol            0
dtype: int64


In [5]:
df['health_condition'] = df['health_condition'].map({'at-risk':2,'unhealthy':1,'fit':0})
X = df.drop('health_condition',axis=1)
y = df['health_condition']


In [6]:
To_Scale = []
To_OnHot = []
To_Ordinal = []
To_Label = []
To_Ordinal_categories = []

for col in X.columns:
    if X[col].dtype in ['object', 'string']:
        print(f"\n==========================================")
        print(f"--- Column Name: {col} ---")
        
        unique_vals = X[col].unique().tolist()
        print("Unique values found:", unique_vals)
        
        while True:
            boss_choice = input('boss is column ko kya kre? \n1 => Onhot \n2 => Ordinal \n3 => Label\n ==========> ')
            
            if boss_choice == '1':
                To_OnHot.append(col)
                break
                
            elif boss_choice == '2':
                To_Ordinal.append(col)
                print(f"\n⚠️ Ordinal ke liye sequence (order) zaroori hai.")
                print(f"Computer ne yeh values dhoondi hain: {unique_vals}")
                
                boss_input = input("Boss, sahi order (Low se High) comma (,) laga kar type karein:\n=====> ")
                
                final_order = [val.strip() for val in boss_input.split(',')]
                
                To_Ordinal_categories.append(final_order)
                break
                
            elif boss_choice == '3':
                To_Label.append(col)
                break
            else:
                print('❌ Samajh nahi aayi boss, dobara sahi option (1, 2, ya 3) likhein.')
                
    elif X[col].dtype in ['int64', 'float64']:
        To_Scale.append(col)

print("\n🎉 MUBARAK HO BOSS! Pura workflow machine-ready hai:")
print("🔹 To_Scale (Numerical)    :", To_Scale)
print("🔹 To_OnHot (Categorical)  :", To_OnHot)
print("🔹 To_Ordinal (Ordered)    :", To_Ordinal)
print("🔹 To_Ordinal_categories   :", To_Ordinal_categories)
print("🔹 To_Label (Extra Labels) :", To_Label)


--- Column Name: stress_level ---
Unique values found: ['high', 'low', 'medium']

⚠️ Ordinal ke liye sequence (order) zaroori hai.
Computer ne yeh values dhoondi hain: ['high', 'low', 'medium']

--- Column Name: sleep_quality ---
Unique values found: ['average', 'poor', 'good']

⚠️ Ordinal ke liye sequence (order) zaroori hai.
Computer ne yeh values dhoondi hain: ['average', 'poor', 'good']

--- Column Name: physical_activity_level ---
Unique values found: ['sedentary', 'moderate', 'active']

⚠️ Ordinal ke liye sequence (order) zaroori hai.
Computer ne yeh values dhoondi hain: ['sedentary', 'moderate', 'active']

--- Column Name: smoking_alcohol ---
Unique values found: ['yes', 'occasional', 'no']

🎉 MUBARAK HO BOSS! Pura workflow machine-ready hai:
🔹 To_Scale (Numerical)    : ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
🔹 To_OnHot (Categorical)  : ['smoking_alcohol']
🔹 To_Ordinal (Ordered)    : ['stress_level', 'sle

In [7]:
OneHot_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False,drop='first')) 
])

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
Ordinal_transformer = Pipeline(steps = [
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('Ordinal',OrdinalEncoder(categories=To_Ordinal_categories))
])
preprocessor = ColumnTransformer([
    ('scaler',numeric_transformer,To_Scale),
    ('onhot',OneHot_transformer,To_OnHot),
    ('ordinal',Ordinal_transformer,To_Ordinal)
])

In [8]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
x_train_sample = x_train.sample(n=100000, random_state=42)
y_train_sample = y_train.loc[x_train_sample.index]

In [ ]:
models = {
    'LogisticRegression':LogisticRegression(max_iter=1200,random_state = 42),
    'RandomForest': RandomForestClassifier(),
    }
parameters = {
    'LogisticRegression':{
        'solver':  ['newton-cholesky'],
        'C': [30],
        'penalty': [ 'l2']
    },
    
    
    'RandomForest':{
        'n_estimators': [200],
        'max_depth': [ 15],
        'min_samples_split': [10],
        'min_samples_leaf': [4],
        'max_features': ['log2'],
   
    },
    'XG':{
    
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 7, 10],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.6, 0.8, 1.0],

    }
}

In [ ]:
best_model={}
for model_name in models:
    print(f'wait .. {model_name} is being trained ...')
    my_pipeline = Pipeline([('preprocessor' , preprocessor),
                            ('model' , models[model_name])
                          ])
    current_grid = {}
    for prams, val in parameters[model_name].items(): 
        current_grid[f'model__{prams}'] = val
    search = RandomizedSearchCV(
        estimator = my_pipeline,
        param_distributions = current_grid,
        cv = 5,
        n_jobs = -1,
        n_iter = 7,
        scoring = 'f1_weighted'
    )
    search.fit(x_train_sample, y_train_sample)
    best_model[model_name] = search.best_estimator_
    print(f"Optimal Parameters for {model_name}: {search.best_params_}")
    print(f"Highest CV Score: {search.best_score_:.4f}\n")


wait .. LogisticRegression is being trained ...
Optimal Parameters for LogisticRegression: {'model__solver': 'newton-cholesky', 'model__penalty': 'l2', 'model__C': 30}
Highest CV Score: 0.9448

wait .. RandomForest is being trained ...
Optimal Parameters for RandomForest: {'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_features': 'log2', 'model__max_depth': 15}
Highest CV Score: 0.9622

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, f1_score,confusion_matrix,precision_score,classification_report

print("======= FINAL TEST SCORES =======")

for model_name, trained_model in best_model.items():

    prediction = trained_model.predict(x_test)

    print(f"\n{model_name}")
    print("-" * 30)
    print("confusion_matrix :", confusion_matrix(y_test, prediction))
    print("Accuracy :", accuracy_score(y_test, prediction))
    print("Precision:", precision_score(y_test, prediction,average='weighted'))
    print("Recall   :", recall_score(y_test, prediction,average='weighted'))
    print("F1 Score :", f1_score(y_test, prediction,average='weighted'))
    print("classification_report :", classification_report(y_test, prediction))

======= FINAL TEST SCORES =======

LogisticRegression
------------------------------
confusion_matrix : [[  5602     21   2336]
 [    13   8174   3251]
 [   760    984 116877]]
Accuracy : 0.9466373951223753
Precision: 0.9447240226926741
Recall   : 0.9466373951223753
F1 Score : 0.9441131615781281
classification_report :               precision    recall  f1-score   support

           0       0.88      0.70      0.78      7959
           1       0.89      0.71      0.79     11438
           2       0.95      0.99      0.97    118621

    accuracy                           0.95    138018
   macro avg       0.91      0.80      0.85    138018
weighted avg       0.94      0.95      0.94    138018


RandomForest
------------------------------
confusion_matrix : [[  6188     26   1745]
 [    14   8639   2785]
 [   245     69 118307]]
Accuracy : 0.9646133113072208
Precision: 0.965086612391642
Recall   : 0.9646133113072208
F1 Score : 0.9627424876135905
classification_report :               prec

======= FINAL TEST SCORES =======

LogisticRegression
------------------------------
confusion_matrix : 
[[  5602     21   2336]
 [    13   8174   3251]
 [   760    984 116877]]
Accuracy : 0.9466373951223753
Precision: 0.9447240226926741
Recall   : 0.9466373951223753
F1 Score : 0.9441131615781281
classification_report :               precision    recall  f1-score   support

           0       0.88      0.70      0.78      7959
           1       0.89      0.71      0.79     11438
           2       0.95      0.99      0.97    118621

    accuracy                           0.95    138018
   macro avg       0.91      0.80      0.85    138018
weighted avg       0.94      0.95      0.94    138018


RandomForest
------------------------------
confusion_matrix : [[  6188     26   1745]
 [    14   8639   2785]
 [   245     69 118307]]
Accuracy : 0.9646133113072208
Precision: 0.965086612391642
Recall   : 0.9646133113072208
F1 Score : 0.9627424876135905
classification_report :               precision    recall  f1-score   support

           0       0.96      0.78      0.86      7959
           1       0.99      0.76      0.86     11438
           2       0.96      1.00      0.98    118621

    accuracy                           0.96    138018
   macro avg       0.97      0.84      0.90    138018
weighted avg       0.97      0.96      0.96    138018



In [ ]:
final_model = best_model['RandomForest']
final_model.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['sleep_duration','heart_rate','bmi',...,'sleep_quality', 'physical_activity_level','smoking_alcohol']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scaler', ...), ('onhot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By s

In [21]:
preddf = pd.read_csv("test.csv")

preds = my_pipeline.predict(preddf)

submission = pd.DataFrame({
    "id": preddf["id"],
    "target": preds
})


In [22]:
submission['target'] = submission['target'].map({2:'at-risk',1:'unhealthy',0:'fit'})
submission.to_csv('prediction.csv',index=False)


# Health Condition Prediction Using Machine Learning

## Project Overview
This project focuses on predicting an individual's **Health Condition** based on lifestyle, health, and activity-related features. The goal is to build a reliable machine learning pipeline that can accurately classify health conditions into multiple categories.

## Data Preparation
The dataset was first cleaned by handling missing values and organizing features into different groups. Numerical features were scaled, categorical features were encoded, and ordinal features were transformed using their natural order. This ensured that all data was ready for machine learning algorithms.

## Machine Learning Pipeline
A complete preprocessing and modeling workflow was built using Scikit-Learn Pipelines and Column Transformers. This approach keeps the entire process organized, reproducible, and free from data leakage.

The following models were trained and compared:

- Logistic Regression
- Random Forest
- XGBoost

Hyperparameter tuning was performed using **RandomizedSearchCV** to find the best-performing configuration for each model.

## Model Evaluation
The models were evaluated using multiple metrics, including:

- Accuracy
- Precision
- Recall
- F1-Score
- Confusion Matrix

Among all tested models, **Random Forest** achieved the best overall performance and was selected as the final model.

## Key Insights
Feature importance analysis revealed that the model relies heavily on:

- Stress Level
- Sleep Duration
- Physical Activity Level

while features such as **ID**, **Gender**, and some dietary categories contributed very little to the final predictions.

## Conclusion
The final machine learning pipeline successfully predicts health conditions with strong performance. The workflow can be reused for new unseen data and provides useful insights into the factors that most influence an individual's health status.